# ML Factory - Google Colab Training

Train all 12 production ML models on financial time-series data using Colab GPU.

## Quick Start
1. **Runtime > Change runtime type > GPU** (T4 or better)
2. **Run Cell 1** (Setup) - clones repo, installs dependencies
3. **Edit Cell 2** (Configuration) - pick models, epochs, features
4. **Run remaining cells** - pipeline runs end-to-end

## Models Available (all 12 confirmed working)

| Category | Models | GPU Benefit |
|----------|--------|-------------|
| **Boosting** | XGBoost, LightGBM, CatBoost | Minimal (fast on CPU) |
| **Neural RNN** | LSTM, GRU | High |
| **Neural CNN** | TCN, InceptionTime, ResNet1D | High |
| **Transformer** | PatchTST, iTransformer, TFT | Critical (TFT needs GPU) |
| **MLP** | N-BEATS | Moderate |

## Features
- **Per-model feature selection** - each model gets its own optimal feature subset
- **4D multi-stream data** - PatchTST/iTransformer get multi-timeframe OHLCV
- **Deploy artifacts** - JSON manifest with correct model names and best-model selection
- **Walk-forward validation** - sliding window training for realistic evaluation
- **Probability calibration** - sigmoid calibration on all model outputs
- **Conformal prediction** - prediction sets with coverage guarantees
- **Leakage detection** - automated checks for data leakage before training

## Data

**Default:** MES (Micro E-mini S&P 500) 1-minute bars (included: `data/raw/MES_1m_1week.parquet`)

**Bring your own data:** Place a `.parquet` or `.csv` file in `data/raw/` and update `DATA_PATH` in Cell 2. Required columns:

| Column | Description |
|--------|-------------|
| `datetime` | Timestamp (index or column) |
| `open` | Open price |
| `high` | High price |
| `low` | Low price |
| `close` | Close price |
| `volume` | Trade volume |

For Colab, upload via the file browser (left panel) or mount Google Drive.

In [ ]:
# =============================================================
# CELL 1: SETUP - Run this first
# =============================================================
import os
import sys
import shutil

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

if IN_COLAB:
    REPO_DIR = "/content/research"

    # Fresh clone
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)

    !git clone https://github.com/Snehpatel101/research.git {REPO_DIR}

    # Install only what Colab doesn't have
    !pip install -q -r {REPO_DIR}/requirements-colab.txt 2>&1 | tail -3

    sys.path.insert(0, REPO_DIR)
    os.chdir(REPO_DIR)

else:
    # Local: assume running from repo root or notebooks/
    REPO_DIR = os.path.dirname(os.path.abspath("."))
    if not os.path.exists(os.path.join(REPO_DIR, "src", "factory.py")):
        REPO_DIR = os.getcwd()
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

# Verify core imports
try:
    from src.factory import MLFactory
    from src.config.experiment import ExperimentConfig
    print("ML Factory loaded successfully!")
except ImportError as e:
    print(f"Import error: {e}")
    raise

# GPU check
import torch

if hasattr(torch, "__version__"):
    print(f"PyTorch: {torch.__version__}")
    if tuple(int(x) for x in torch.__version__.split(".")[:2]) < (2, 0):
        print("WARNING: PyTorch < 2.0 detected. Some models may not work correctly.")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("No GPU detected. Boosting models OK, neural models will be slow.")
    if IN_COLAB:
        print(">> Runtime > Change runtime type > GPU")

print(f"\nRepo: {REPO_DIR}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

In [ ]:
# =============================================================
# CELL 2: CONFIGURATION - Edit these settings
# =============================================================

# ═══════════════════════════════════════════════════════════════
# PRESETS (uncomment one block to override defaults below)
# ═══════════════════════════════════════════════════════════════
# Quick Test (~2 min, CPU OK):
#   MODELS = ["xgboost", "lightgbm"]; MAX_EPOCHS = 3; N_SPLITS = 2
#   TRAINING_MODE = "standard"; FEATURE_SELECTION_ENABLED = False
#
# Boosting Suite (~5 min):
#   MODELS = ["xgboost", "lightgbm", "catboost"]; N_SPLITS = 5
#   MAX_EPOCHS = 50; FEATURE_SELECTION_ENABLED = True
#
# Full GPU (~30 min, all 12 models):
#   MODELS = ["xgboost","lightgbm","catboost","lstm","gru","tcn",
#             "inceptiontime","resnet1d","patchtst","itransformer","tft","nbeats"]
#   MAX_EPOCHS = 50; N_SPLITS = 5; TRAINING_MODE = "standard"
#
# Walk-Forward Evaluation (~15 min):
#   MODELS = ["xgboost", "lightgbm", "catboost"]
#   TRAINING_MODE = "walk_forward"; WF_N_WINDOWS = 5
#   WF_WINDOW_TYPE = "expanding"; WF_MIN_TRAIN_PCT = 0.4
# ═══════════════════════════════════════════════════════════════

# --- DATA ---
SYMBOL = "MES"
DATA_PATH = f"{REPO_DIR}/data/raw/MES_1m_1week.parquet"
TARGET_TIMEFRAME = "5min"

# --- MODELS (toggle True/False) ---
# Boosting (fast, ~1-2 min each)
USE_XGBOOST = True
USE_LIGHTGBM = True
USE_CATBOOST = True

# Neural RNN (moderate, ~5-8 min with GPU)
USE_LSTM = True
USE_GRU = True

# Neural CNN (moderate-heavy, 5-30 min with GPU)
USE_TCN = True
USE_INCEPTIONTIME = True
USE_RESNET1D = True

# Transformers (heavy, need GPU)
USE_PATCHTST = True
USE_ITRANSFORMER = True
USE_TFT = True          # Slowest model - ~10h on CPU, ~30min on GPU

# MLP
USE_NBEATS = True

# --- TRAINING ---
HORIZONS = [5, 10]                  # Prediction horizons (bars ahead) — supports multiple horizons
N_SPLITS = 2                        # CV folds (2-3 for small data, 5 for large)
PURGE_BARS = 10                     # Purge bars between train/test
EMBARGO_BARS = 60                   # Embargo bars after test set
MAX_EPOCHS = 50                     # 3=quick test, 50=decent, 100=full
EARLY_STOPPING_PATIENCE = 10       # Stop if no improvement for N epochs
BATCH_SIZE = 256
DEVICE = "auto"                     # auto picks GPU if available

# --- WALK-FORWARD VALIDATION ---
TRAINING_MODE = "standard"          # "standard" = single split, "walk_forward" = sliding windows
WF_N_WINDOWS = 5                    # Number of test windows
WF_WINDOW_TYPE = "expanding"        # "expanding" = growing train, "rolling" = fixed-size train
WF_MIN_TRAIN_PCT = 0.4              # Minimum training data (40% of total)
WF_TEST_PCT = 0.1                   # Test data per window (10% of total)

# --- FEATURES ---
MTF_ENABLED = True                  # Multi-timeframe features
MTF_TIMEFRAMES = ["15min", "30min", "1h"]

# Feature selection — MDA-based per-model feature importance filtering
FEATURE_SELECTION_ENABLED = False   # Enable per-model feature selection
FEATURE_SELECTION_METHOD = "mda"    # mda (permutation importance), mdi, hybrid
FEATURE_SELECTION_N_FEATURES = 50   # Max features to keep per model
FEATURE_SELECTION_CV_SPLITS = 5     # CV folds for feature importance scoring
FEATURE_SELECTION_MIN_FREQUENCY = 0.6  # Min selection frequency across CV folds (0.0-1.0)

# --- TRANSACTION COSTS (used by backtest & financial report) ---
COMMISSION_PER_TRADE = 2.50         # Per side, in dollars
SLIPPAGE_TICKS = 1.0                # Slippage in ticks per trade
TICK_VALUE = 1.25                   # Dollar value per tick (MES=1.25, MNQ=0.50)
INITIAL_EQUITY = 100000.0           # Starting portfolio value

# --- CONFORMAL PREDICTION ---
CONFORMAL_ENABLED = False           # Prediction sets with coverage guarantees
CONFORMAL_ALPHA = 0.1               # Miscoverage rate (0.1 = 90% coverage)
CONFORMAL_METHOD = "aps"            # "aps" (adaptive), "lac" (least ambiguous)

# --- ENSEMBLE ---
BUILD_ENSEMBLE = True
META_LEARNER = "ridge_meta"         # ridge_meta, mlp_meta, xgboost_meta

# --- OPTUNA ---
OPTUNA_ENABLED = False              # Set True + OPTUNA_TRIALS for hyperparameter tuning
OPTUNA_TRIALS = 50                  # 0=disable, 25=quick, 50=balanced
OPTUNA_N_STARTUP_TRIALS = 10       # Random trials before TPE sampler kicks in
OPTUNA_MAX_TIME = None              # Max optimization time in seconds (None = no limit)

# --- EVALUATION ---
RUN_BACKTEST = True                 # Backtest with equity curve, Sharpe, drawdown, profit factor
GENERATE_REPORT = True

# --- BUNDLING ---
CREATE_BUNDLE = True                # Create inference bundles for each model
DEPLOY_ARTIFACT = True              # Create deploy manifest indexing all bundles

# --- EXPERIMENT ---
EXPERIMENT_NAME = "mes_e2e_12_models"
RANDOM_SEED = 42

# =============================================================
# BUILD MODEL LIST (auto from toggles above)
# =============================================================
MODELS = []
if USE_XGBOOST: MODELS.append("xgboost")
if USE_LIGHTGBM: MODELS.append("lightgbm")
if USE_CATBOOST: MODELS.append("catboost")
if USE_LSTM: MODELS.append("lstm")
if USE_GRU: MODELS.append("gru")
if USE_TCN: MODELS.append("tcn")
if USE_INCEPTIONTIME: MODELS.append("inceptiontime")
if USE_RESNET1D: MODELS.append("resnet1d")
if USE_PATCHTST: MODELS.append("patchtst")
if USE_ITRANSFORMER: MODELS.append("itransformer")
if USE_TFT: MODELS.append("tft")
if USE_NBEATS: MODELS.append("nbeats")

print(f"Models: {len(MODELS)} selected -> {MODELS}")
print(f"Epochs: {MAX_EPOCHS}, Horizons: {HORIZONS}, Device: {DEVICE}")
print(f"Training mode: {TRAINING_MODE}" + (f" ({WF_N_WINDOWS} windows, {WF_WINDOW_TYPE})" if TRAINING_MODE == "walk_forward" else ""))
print(f"Feature selection: {'enabled' if FEATURE_SELECTION_ENABLED else 'disabled'}" + (f" ({FEATURE_SELECTION_METHOD}, top {FEATURE_SELECTION_N_FEATURES}, min_freq={FEATURE_SELECTION_MIN_FREQUENCY})" if FEATURE_SELECTION_ENABLED else ""))
print(f"Ensemble: {BUILD_ENSEMBLE}, Optuna trials: {OPTUNA_TRIALS if OPTUNA_ENABLED else 'disabled'}" + (f" (startup={OPTUNA_N_STARTUP_TRIALS}, max_time={OPTUNA_MAX_TIME})" if OPTUNA_ENABLED else ""))
print(f"Transaction costs: commission=${COMMISSION_PER_TRADE}/side, slippage={SLIPPAGE_TICKS} ticks (${SLIPPAGE_TICKS * TICK_VALUE:.2f})")
print(f"Conformal prediction: {'enabled' if CONFORMAL_ENABLED else 'disabled'}" + (f" ({CONFORMAL_METHOD}, alpha={CONFORMAL_ALPHA})" if CONFORMAL_ENABLED else ""))
print(f"Bundling: create={CREATE_BUNDLE}, deploy={DEPLOY_ARTIFACT}")

In [ ]:
# =============================================================
# CELL 3: VALIDATE CONFIGURATION
# =============================================================
import os
import torch

VALID_MODELS = {
    "xgboost", "lightgbm", "catboost",
    "lstm", "gru", "tcn", "nbeats",
    "inceptiontime", "resnet1d",
    "patchtst", "itransformer", "tft",
}
NEURAL_MODELS = {
    "lstm", "gru", "tcn", "nbeats",
    "inceptiontime", "resnet1d",
    "patchtst", "itransformer", "tft",
}

errors, warnings = [], []

# Model check
for m in MODELS:
    if m not in VALID_MODELS:
        errors.append(f"Unknown model: '{m}'")
if not MODELS:
    errors.append("No models selected!")

# Data check
if not os.path.exists(DATA_PATH):
    errors.append(f"Data file not found: {DATA_PATH}")

# Training mode check
if TRAINING_MODE not in ("standard", "walk_forward", "regime_aware", "meta_labeling"):
    errors.append(f"Unknown training mode: '{TRAINING_MODE}'")

if TRAINING_MODE == "walk_forward":
    if WF_MIN_TRAIN_PCT + WF_N_WINDOWS * WF_TEST_PCT > 1.0:
        errors.append(
            f"Walk-forward config invalid: min_train_pct ({WF_MIN_TRAIN_PCT}) + "
            f"n_windows ({WF_N_WINDOWS}) * test_pct ({WF_TEST_PCT}) > 1.0"
        )
    print(f"Walk-forward: {WF_N_WINDOWS} windows, {WF_WINDOW_TYPE}, "
          f"min_train={WF_MIN_TRAIN_PCT*100:.0f}%, test={WF_TEST_PCT*100:.0f}% per window")

# GPU check for neural models
selected_neural = [m for m in MODELS if m in NEURAL_MODELS]
if selected_neural and not torch.cuda.is_available():
    warnings.append(
        f"No GPU but neural models selected: {selected_neural}. "
        "Will be slow. Runtime > Change runtime type > GPU"
    )

if "tft" in MODELS and not torch.cuda.is_available():
    warnings.append("TFT without GPU will take ~10 hours. Consider disabling it.")

# Memory warning for walk-forward + many models
if TRAINING_MODE == "walk_forward" and len(MODELS) >= 8:
    est_gb = len(MODELS) * WF_N_WINDOWS * 0.5
    warnings.append(
        f"Walk-forward with {len(MODELS)} models x {WF_N_WINDOWS} windows "
        f"may need ~{est_gb:.0f} GB RAM. "
        "If OOM: reduce models, windows, or use 'standard' mode first."
    )

# Report
if errors:
    for e in errors:
        print(f"ERROR: {e}")
    raise ValueError("Fix errors above in Cell 2")

if warnings:
    for w in warnings:
        print(f"WARNING: {w}")

print(f"\nConfig OK: {len(MODELS)} models, {N_SPLITS} CV folds, "
      f"purge={PURGE_BARS}, embargo={EMBARGO_BARS}, device={DEVICE}, mode={TRAINING_MODE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [4]:
# =============================================================
# CELL 4: LOAD & PREVIEW DATA
# =============================================================
import pandas as pd

# Load data based on file extension
if DATA_PATH.endswith(".parquet"):
    raw_data = pd.read_parquet(DATA_PATH)
elif DATA_PATH.endswith(".csv"):
    raw_data = pd.read_csv(DATA_PATH)
else:
    raise ValueError(f"Unsupported file format: {DATA_PATH}. Use .parquet or .csv")

# Normalize column names to lowercase
raw_data.columns = [c.lower().strip() for c in raw_data.columns]

# Validate required OHLCV columns
REQUIRED_COLUMNS = ["open", "high", "low", "close", "volume"]
missing = [c for c in REQUIRED_COLUMNS if c not in raw_data.columns]
if missing:
    raise ValueError(
        f"Missing required OHLCV columns: {missing}\n"
        f"Found columns: {list(raw_data.columns)}\n"
        f"The pipeline expects: {REQUIRED_COLUMNS}"
    )

# Ensure datetime index
if "datetime" in raw_data.columns:
    raw_data["datetime"] = pd.to_datetime(raw_data["datetime"])
    raw_data = raw_data.set_index("datetime").sort_index()
elif "date" in raw_data.columns:
    raw_data["date"] = pd.to_datetime(raw_data["date"])
    raw_data = raw_data.set_index("date").sort_index()
    raw_data.index.name = "datetime"
elif not isinstance(raw_data.index, pd.DatetimeIndex):
    # Try parsing the existing index as datetime
    try:
        raw_data.index = pd.to_datetime(raw_data.index)
        raw_data.index.name = "datetime"
        raw_data = raw_data.sort_index()
    except Exception:
        raise ValueError(
            "Could not find or parse a datetime column. "
            "Data must have a 'datetime' or 'date' column, or a datetime-parseable index."
        )

# --- Summary ---
print("=" * 50)
print("Data Loaded Successfully")
print("=" * 50)
print(f"  Symbol:      {SYMBOL}")
print(f"  Rows:        {len(raw_data):,}")
print(f"  Shape:       {raw_data.shape}")
print(f"  Columns:     {list(raw_data.columns)}")
print(f"  Date range:  {raw_data.index.min()} -> {raw_data.index.max()}")
print(f"  Index name:  {raw_data.index.name}")
print()

# Missing values
missing_counts = raw_data[REQUIRED_COLUMNS].isnull().sum()
total_missing = missing_counts.sum()
if total_missing > 0:
    print("WARNING: Missing values in OHLCV columns:")
    for col, count in missing_counts.items():
        if count > 0:
            print(f"  {col}: {count} ({count/len(raw_data)*100:.2f}%)")
else:
    print("No missing values in OHLCV columns.")
print()

# Preview
print("First 5 rows:")
display(raw_data.head())

print(f"\nData ready: 'raw_data' DataFrame with {len(raw_data):,} rows.")

## Exploratory Data Analysis

Visual inspection of the loaded data before running the pipeline. Checks for price patterns, volume distribution, return characteristics, and session gaps.

In [ ]:
# =============================================================
# EDA: Price, Volume, Returns & Data Quality
# =============================================================
try:
    import matplotlib.pyplot as plt
    import matplotlib.dates as mdates
    import numpy as np
    %matplotlib inline

    df = raw_data.copy()

    # --- Figure 1: Price + Volume + Returns Distribution ---
    fig, axes = plt.subplots(3, 1, figsize=(14, 10), gridspec_kw={'height_ratios': [3, 1, 1]})

    # Price chart
    axes[0].plot(df.index, df['close'], linewidth=0.8, color='#1f77b4')
    axes[0].set_title(f'{SYMBOL} Close Price — {df.index[0]:%Y-%m-%d %H:%M} to {df.index[-1]:%Y-%m-%d %H:%M}', fontsize=13)
    axes[0].set_ylabel('Price')
    axes[0].grid(True, alpha=0.3)

    # Volume
    axes[1].bar(df.index, df['volume'], width=0.001, alpha=0.5, color='#2ca02c')
    axes[1].set_title('Volume', fontsize=11)
    axes[1].set_ylabel('Volume')
    axes[1].grid(True, alpha=0.3)

    # Returns distribution
    returns = df['close'].pct_change().dropna()
    axes[2].hist(returns, bins=100, alpha=0.7, edgecolor='black', linewidth=0.3, color='#ff7f0e')
    skew_val = returns.skew()
    kurt_val = returns.kurtosis()
    axes[2].set_title(f'Returns Distribution (skew={skew_val:.3f}, kurtosis={kurt_val:.2f})', fontsize=11)
    axes[2].set_xlabel('Return')
    axes[2].set_ylabel('Count')
    axes[2].axvline(0, color='red', linestyle='--', alpha=0.5)
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # --- Data Quality Stats ---
    print("=" * 50)
    print("DATA QUALITY SUMMARY")
    print("=" * 50)
    print(f"  Bars:          {len(df):,}")
    print(f"  Date range:    {df.index.min()} -> {df.index.max()}")
    duration = df.index.max() - df.index.min()
    print(f"  Duration:      {duration}")
    print(f"  Missing OHLCV: {df[['open','high','low','close','volume']].isnull().sum().sum()}")

    # Gap detection (gaps > 4 hours suggest session breaks or missing data)
    time_diffs = df.index.to_series().diff()
    large_gaps = time_diffs[time_diffs > pd.Timedelta(hours=4)].dropna()
    print(f"\n  Gaps > 4h:     {len(large_gaps)}")
    if len(large_gaps) > 0 and len(large_gaps) <= 20:
        for ts, gap in large_gaps.items():
            print(f"    {ts}: {gap}")
    elif len(large_gaps) > 20:
        print(f"    (showing first 10 of {len(large_gaps)})")
        for ts, gap in list(large_gaps.items())[:10]:
            print(f"    {ts}: {gap}")

    # Basic return stats
    print(f"\n  Return Stats (1-bar):")
    print(f"    Mean:     {returns.mean():.6f}")
    print(f"    Std:      {returns.std():.6f}")
    print(f"    Skew:     {skew_val:.4f}")
    print(f"    Kurtosis: {kurt_val:.4f}")
    print(f"    Min:      {returns.min():.6f}")
    print(f"    Max:      {returns.max():.6f}")

except Exception as e:
    print(f"EDA skipped: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# =============================================================
# CELL 5: RUN ML FACTORY
# =============================================================
from src.config.experiment import (
    ExperimentConfig,
    DataSection,
    TrainingSection,
    EvaluationSection,
    BundlingSection,
)
from src.config.training import OptunaConfig
from src.config.data import FeatureConfig, LabelingConfig, MTFConfig
from src.config.cv import WalkForwardConfig
from src.factory import MLFactory

config = ExperimentConfig(
    name=EXPERIMENT_NAME,
    random_seed=RANDOM_SEED,
    verbose=1,

    data=DataSection(
        symbol=SYMBOL,
        data_path=DATA_PATH,
        features=FeatureConfig(
            mode="full",
            selection_enabled=FEATURE_SELECTION_ENABLED,
            selection_method=FEATURE_SELECTION_METHOD if FEATURE_SELECTION_ENABLED else "mda",
            selection_n_features=FEATURE_SELECTION_N_FEATURES,
            selection_cv_splits=FEATURE_SELECTION_CV_SPLITS,
        ),
        labeling=LabelingConfig(method="triple_barrier"),
        mtf=MTFConfig(
            enabled=MTF_ENABLED,
            mode="indicators" if MTF_ENABLED else "none",
            timeframes=MTF_TIMEFRAMES if MTF_ENABLED else [],
            primary_timeframe=TARGET_TIMEFRAME,
        ),
    ),

    training=TrainingSection(
        models=MODELS,
        horizons=HORIZONS,
        training_mode=TRAINING_MODE,
        n_splits=N_SPLITS,
        purge_bars=PURGE_BARS,
        embargo_bars=EMBARGO_BARS,
        device=DEVICE,
        batch_size=BATCH_SIZE,
        max_epochs=MAX_EPOCHS,
        early_stopping_patience=EARLY_STOPPING_PATIENCE,
        build_ensemble=BUILD_ENSEMBLE,
        meta_learner=META_LEARNER,
        # Walk-forward validation config (used when TRAINING_MODE="walk_forward")
        walk_forward=WalkForwardConfig(
            n_windows=WF_N_WINDOWS,
            window_type=WF_WINDOW_TYPE,
            min_train_pct=WF_MIN_TRAIN_PCT,
            test_pct=WF_TEST_PCT,
            embargo_bars=EMBARGO_BARS,
            gap_bars=PURGE_BARS,
        ),
        optuna=OptunaConfig(
            n_trials=OPTUNA_TRIALS if OPTUNA_ENABLED else 0,
            n_startup_trials=OPTUNA_N_STARTUP_TRIALS,
            timeout=OPTUNA_MAX_TIME if OPTUNA_MAX_TIME is not None else 0,
        ),
    ),

    evaluation=EvaluationSection(
        run_backtest=RUN_BACKTEST,
        generate_report=GENERATE_REPORT,
    ),

    bundling=BundlingSection(
        create_bundle=CREATE_BUNDLE,
        deploy_artifact=DEPLOY_ARTIFACT,
    ),
)

print(f"Experiment: {config.name}")
print(f"Models: {config.training.models}")
print(f"Training mode: {config.training.training_mode}")
if config.training.training_mode == "walk_forward":
    wf = config.training.walk_forward
    print(f"Walk-forward: {wf.n_windows} windows, {wf.window_type}, min_train={wf.min_train_pct*100:.0f}%, test={wf.test_pct*100:.0f}%")
print(f"CV: {config.training.n_splits} folds, purge={config.training.purge_bars}, embargo={config.training.embargo_bars}")
print(f"Epochs: {config.training.max_epochs}, Device: {config.training.device}")
print(f"Feature selection: enabled={config.data.features.selection_enabled}, method={config.data.features.selection_method}, n_features={config.data.features.selection_n_features}, cv_splits={config.data.features.selection_cv_splits}")
print(f"Optuna: {'enabled' if OPTUNA_ENABLED else 'disabled'} ({config.training.optuna.n_trials} trials, startup={config.training.optuna.n_startup_trials}, timeout={config.training.optuna.timeout}s)")
print(f"Bundling: create={config.bundling.create_bundle}, deploy={config.bundling.deploy_artifact}")
print()

factory = MLFactory(config, enable_checkpoints=True)

try:
    result = factory.run()
    print()
    if result.success:
        print(result.summary())
    else:
        print(f"Pipeline completed with errors: {result.error_message}")
except KeyboardInterrupt:
    print("\nInterrupted. Resume with: result = factory.resume_from_checkpoint()")
    result = None
except Exception as e:
    print(f"\nERROR: {e}")
    import traceback
    traceback.print_exc()
    print("\nResume with: result = factory.resume_from_checkpoint()")
    result = None

In [ ]:
# =============================================================
# CELL 6: RESULTS & VISUALIZATION
# =============================================================
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
import glob

if "result" not in dir() or result is None or not result.success:
    msg = "No successful result to display."
    if "result" in dir() and result and result.error_message:
        msg += f"\nError: {result.error_message}"
    print(msg)
else:
    print("=" * 60)
    print("EXPERIMENT RESULTS")
    print("=" * 60)
    print(f"  Run ID:          {result.run_id}")
    print(f"  Models trained:  {result.n_models}")
    print(f"  Best model:      {result.best_model}")
    print(f"  Duration:        {result.duration_seconds:.1f}s ({result.duration_seconds/60:.1f} min)")
    print()

    # --- Model Metrics Table ---
    if result.metrics:
        print("-" * 40)
        print("Model Performance")
        print("-" * 40)
        metrics_df = pd.DataFrame(result.metrics).T
        metrics_df.index.name = "model"
        display(metrics_df.round(4))
        print()

    # --- Ensemble Metrics ---
    if result.ensemble_metrics:
        print("-" * 40)
        print("Ensemble Metrics")
        print("-" * 40)
        for k, v in result.ensemble_metrics.items():
            if isinstance(v, float):
                print(f"  {k}: {v:.4f}")
            else:
                print(f"  {k}: {v}")
        print()

    # --- Backtest Metrics ---
    if result.backtest_metrics:
        print("-" * 40)
        print("Backtest Results")
        print("-" * 40)
        highlight_keys = ["sharpe_ratio", "max_drawdown_pct", "profit_factor", "win_rate_pct"]
        for k in highlight_keys:
            if k in result.backtest_metrics:
                print(f"  {k}: {result.backtest_metrics[k]}")
        for k, v in result.backtest_metrics.items():
            if k not in highlight_keys:
                if isinstance(v, (int, float)):
                    print(f"  {k}: {v}")
        print()

    # --- Display Plot Images ---
    if result.output_dir and Path(result.output_dir).exists():
        plot_files = sorted(glob.glob(str(Path(result.output_dir) / "**" / "*.png"), recursive=True))[:6]
        if plot_files:
            print("-" * 40)
            print(f"Plots ({len(plot_files)} found)")
            print("-" * 40)
            n_plots = len(plot_files)
            cols = min(n_plots, 2)
            rows = (n_plots + cols - 1) // cols
            fig, axes = plt.subplots(rows, cols, figsize=(7 * cols, 5 * rows))
            if n_plots == 1:
                axes = [axes]
            else:
                axes = axes.flatten() if hasattr(axes, "flatten") else [axes]
            for i, pf in enumerate(plot_files):
                img = mpimg.imread(pf)
                axes[i].imshow(img)
                axes[i].set_title(Path(pf).stem, fontsize=10)
                axes[i].axis("off")
            # Hide unused subplots
            for j in range(n_plots, len(axes)):
                axes[j].axis("off")
            plt.tight_layout()
            plt.show()

    if result.bundle_path:
        print(f"Bundle path: {result.bundle_path}")
    if result.output_dir:
        print(f"Output dir:  {result.output_dir}")

## Calibration & Conformal Prediction

Probability calibration quality and conformal prediction sets (coverage guarantees). Requires a successful training run above.

In [ ]:
# =============================================================
# CALIBRATION & CONFORMAL PREDICTION
# =============================================================
try:
    import numpy as np
    import matplotlib.pyplot as plt

    if "result" not in dir() or result is None or not result.success:
        print("No results available. Run the pipeline cell first.")
    else:
        # --- Probability Calibration ---
        # The pipeline applies sigmoid (Platt) calibration automatically.
        # Show calibration reliability diagram if OOF predictions are available.
        cal_shown = False
        if hasattr(result, 'training_result') and result.training_result is not None:
            tr = result.training_result
            oof_data = getattr(tr, 'oof_predictions', None) or getattr(tr, 'aligned_results', None)
            if oof_data and isinstance(oof_data, dict):
                print("=" * 50)
                print("PROBABILITY CALIBRATION (Sigmoid / Platt Scaling)")
                print("=" * 50)
                for model_name, oof in list(oof_data.items())[:4]:
                    probas = getattr(oof, 'probabilities', None)
                    labels = getattr(oof, 'labels', getattr(oof, 'y_true', None))
                    if probas is not None and labels is not None:
                        probas = np.asarray(probas)
                        labels = np.asarray(labels)
                        if probas.ndim == 2 and probas.shape[1] >= 2:
                            # Reliability diagram for class 1 (Long)
                            cls_idx = 2 if probas.shape[1] == 3 else 1
                            p = probas[:, cls_idx]
                            y = (labels == (1 if probas.shape[1] == 3 else cls_idx)).astype(int)
                            n_bins = 10
                            bin_edges = np.linspace(0, 1, n_bins + 1)
                            bin_means, bin_true = [], []
                            for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
                                mask = (p >= lo) & (p < hi)
                                if mask.sum() > 0:
                                    bin_means.append(p[mask].mean())
                                    bin_true.append(y[mask].mean())
                            if bin_means:
                                print(f"  {model_name}: {len(bin_means)} bins with data")
                                cal_shown = True
                print()

        if not cal_shown:
            print("Calibration details: OOF predictions not directly accessible.")
            print("Calibration is applied automatically by the pipeline (sigmoid/Platt scaling).")
            print()

        # --- Conformal Prediction ---
        if CONFORMAL_ENABLED:
            print("=" * 50)
            print(f"CONFORMAL PREDICTION (method={CONFORMAL_METHOD}, alpha={CONFORMAL_ALPHA})")
            print("=" * 50)
            from src.models.calibration.conformal import ConformalPredictor, ConformalConfig

            conf_config = ConformalConfig(
                enabled=True,
                alpha=CONFORMAL_ALPHA,
                method=CONFORMAL_METHOD,
            )
            print(f"  Coverage target: {(1 - CONFORMAL_ALPHA) * 100:.0f}%")
            print(f"  Method: {CONFORMAL_METHOD}")
            print()
            print("  Conformal prediction is configured. When OOF probabilities")
            print("  are available, it produces prediction sets with guaranteed")
            print("  coverage at the specified confidence level.")
            print()
            print("  To use standalone:")
            print("    conformal = ConformalPredictor(config)")
            print("    metrics = conformal.fit(y_cal, probas_cal)")
            print("    sets, sizes = conformal.predict_sets(probas_test)")
        else:
            print("Conformal prediction: disabled. Set CONFORMAL_ENABLED=True in Cell 2.")

except Exception as e:
    print(f"Calibration/conformal section skipped: {e}")
    import traceback
    traceback.print_exc()

## Leakage Detection

Automated checks for data leakage before training. Detects suspiciously high feature-label correlations that may indicate information from the future leaking into features.

In [ ]:
# =============================================================
# LEAKAGE DETECTION: Check for data leakage before/after training
# =============================================================
try:
    from src.validation.leakage_detection import comprehensive_leakage_check
    import numpy as np

    if "result" not in dir() or result is None or not result.success:
        print("No results available. Run the pipeline cell first.")
        print("Leakage detection runs on pipeline features + labels after training.")
    else:
        # Try to extract features and labels from result
        features_df = None
        labels_arr = None

        tr = getattr(result, 'training_result', None)
        if tr is not None:
            features_df = getattr(tr, 'features', getattr(tr, 'feature_matrix', None))
            labels_arr = getattr(tr, 'labels', getattr(tr, 'y', None))

        if features_df is not None and labels_arr is not None:
            labels_arr = np.asarray(labels_arr)
            print("=" * 50)
            print("LEAKAGE DETECTION")
            print("=" * 50)
            print(f"  Features: {features_df.shape}")
            print(f"  Labels:   {labels_arr.shape}")
            print()

            reports = comprehensive_leakage_check(
                features=features_df,
                labels=labels_arr,
                feature_names=list(features_df.columns) if hasattr(features_df, 'columns') else None,
                correlation_threshold=0.5,
            )
            leakage_found = False
            for check_name, report in reports.items():
                if report.leakage_detected:
                    leakage_found = True
                    print(f"  WARNING - {check_name}: {report.message}")
                    if report.suspicious_features:
                        for feat in report.suspicious_features[:5]:
                            print(f"    - {feat}")

            if not leakage_found:
                print("  No leakage detected. All checks passed.")
            print()
        else:
            print("Leakage detection: features/labels not directly accessible from result.")
            print("The pipeline enforces purge/embargo to prevent leakage during training.")
            print(f"  Purge bars: {PURGE_BARS}, Embargo bars: {EMBARGO_BARS}")

except ImportError:
    print("Leakage detection module not available.")
except Exception as e:
    print(f"Leakage detection skipped: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# =============================================================
# CELL 7: DEPLOY ARTIFACT - Production inference entry point
# =============================================================
#
# After training, the factory creates a deploy/ directory with a
# JSON manifest indexing all bundles by horizon. This cell shows
# how to load and use the deploy artifact for inference.
#
# Usage:
#   artifact = load_deploy_artifact("./deploy", horizon=20)
#   pred = artifact.predict_from_raw(raw_bars_df)

from pathlib import Path
import pandas as pd

if "result" not in dir() or result is None or not result.success:
    print("No successful result. Run Cell 5 first.")
elif result.deploy_path and Path(result.deploy_path).exists():
    from src.inference.deploy import (
        load_deploy_artifact,
        validate_deploy_artifact,
        DeployManifest,
        DEPLOY_MANIFEST_FILE,
    )

    deploy_dir = Path(result.deploy_path)

    # --- Validate the deploy artifact ---
    validation = validate_deploy_artifact(deploy_dir)
    print("=" * 50)
    print("DEPLOY ARTIFACT")
    print("=" * 50)
    print(f"  Path:     {deploy_dir}")
    print(f"  Valid:    {validation['valid']}")
    print(f"  Horizons: {validation.get('n_horizons', 0)}")
    if validation["issues"]:
        for issue in validation["issues"]:
            print(f"  ISSUE: {issue}")
    print()

    # --- Show manifest contents ---
    manifest = DeployManifest.load(deploy_dir / DEPLOY_MANIFEST_FILE)
    for h, h_manifest in sorted(manifest.horizons.items()):
        print(f"  Horizon {h}:")
        print(f"    Primary model: {h_manifest.primary_model}")
        for entry in h_manifest.entries:
            tag = " [ensemble]" if entry.is_ensemble else ""
            f1 = entry.metrics.get("macro_f1", 0.0)
            print(f"    - {entry.model_name}{tag} (F1={f1:.4f}) -> {entry.bundle_path}")
    print()

    # --- Load the primary artifact for inference ---
    horizon = HORIZONS[0] if HORIZONS else 20
    try:
        artifact = load_deploy_artifact(deploy_dir, horizon=horizon)
        print(f"Loaded artifact for H{horizon}: {type(artifact).__name__}")
        model_name = getattr(artifact.metadata, "model_name", "N/A")
        print(f"  Model:  {model_name}")
        print(f"  Symbol: {getattr(artifact.metadata, 'symbol', 'N/A')}")
        print()
        print("Ready for inference:")
        print("  pred = artifact.predict_from_raw(raw_bars_df)")
    except Exception as e:
        print(f"Could not load artifact: {e}")
        print("(This is normal if bundles were not saved to deploy dir)")

    # --- Run predict_from_raw() on a sample of raw data ---
    print()
    print("-" * 50)
    print("INFERENCE DEMO: predict_from_raw()")
    print("-" * 50)
    try:
        # Use last 500 bars as a sample (enough for indicator warm-up)
        sample = raw_data.tail(500).copy()
        print(f"  Sample: {len(sample)} bars ({sample.index.min()} -> {sample.index.max()})")

        pred = artifact.predict_from_raw(sample)

        # Extract class predictions
        if hasattr(pred, "class_predictions"):
            classes = pd.Series(pred.class_predictions)
        elif hasattr(pred, "predictions") and hasattr(pred.predictions, "class_predictions"):
            classes = pd.Series(pred.predictions.class_predictions)
        elif isinstance(pred, pd.DataFrame) and "prediction" in pred.columns:
            classes = pred["prediction"]
        else:
            classes = pd.Series(pred) if not isinstance(pred, pd.Series) else pred

        # Class distribution
        label_map = {-1: "Short", 0: "Neutral", 1: "Long"}
        dist = classes.value_counts().sort_index()
        print(f"\n  Predictions: {len(classes)} samples")
        print("  Class distribution:")
        for cls_val, count in dist.items():
            label = label_map.get(int(cls_val), str(cls_val))
            pct = count / len(classes) * 100
            print(f"    {label:>8s} ({int(cls_val):+d}): {count:>5d}  ({pct:5.1f}%)")

        # Confidence stats if available
        if hasattr(pred, "confidence"):
            conf = pred.confidence
        elif hasattr(pred, "predictions") and hasattr(pred.predictions, "confidence"):
            conf = pred.predictions.confidence
        else:
            conf = None
        if conf is not None:
            import numpy as np
            conf = np.asarray(conf)
            print(f"\n  Confidence: mean={conf.mean():.3f}, std={conf.std():.3f}, "
                  f"min={conf.min():.3f}, max={conf.max():.3f}")

        print("\n  Inference demo complete.")

    except Exception as e:
        print(f"  predict_from_raw() failed: {e}")
        print("  (Expected if preprocessing graph was not saved with bundle)")
else:
    print("No deploy artifact found.")
    print("Ensure DEPLOY_ARTIFACT=True in Cell 2 config.")

## Model Comparison

Side-by-side comparison of all trained models across key classification metrics.

In [ ]:
# =============================================================
# MODEL COMPARISON: Bar chart of metrics across all models
# =============================================================
try:
    import matplotlib.pyplot as plt
    import pandas as pd
    import numpy as np

    if "result" not in dir() or result is None or not result.success or not result.metrics:
        print("No model metrics available. Run Cell 5 first.")
    else:
        metrics_df = pd.DataFrame(result.metrics).T
        metrics_df.index.name = "model"

        # Select key metrics (filter to what's actually available)
        key_metrics = ["macro_f1", "accuracy", "precision", "recall", "mcc"]
        available = [m for m in key_metrics if m in metrics_df.columns]
        if not available:
            # Fallback: use whatever numeric columns exist
            available = [c for c in metrics_df.columns if metrics_df[c].dtype in ('float64', 'float32', 'int64')][:5]

        if available:
            plot_df = metrics_df[available].copy()
            plot_df.columns = [c.replace("macro_", "").replace("_", " ").title() for c in plot_df.columns]

            fig, ax = plt.subplots(figsize=(max(12, len(plot_df) * 1.2), 6))
            plot_df.plot(kind='bar', ax=ax, width=0.8, edgecolor='black', linewidth=0.5)
            ax.set_title('Model Comparison — Key Metrics', fontsize=14)
            ax.set_ylabel('Score')
            ax.set_ylim(0, min(1.05, plot_df.max().max() * 1.15))
            ax.legend(loc='lower right', framealpha=0.9)
            ax.grid(True, axis='y', alpha=0.3)
            plt.xticks(rotation=45, ha='right')
            plt.tight_layout()
            plt.show()

            # Print ranking by primary metric
            rank_col = "macro_f1" if "macro_f1" in metrics_df.columns else available[0]
            ranked = metrics_df[rank_col].sort_values(ascending=False)
            print(f"\nRanking by {rank_col}:")
            for i, (model, score) in enumerate(ranked.items(), 1):
                marker = " ◀ best" if i == 1 else ""
                print(f"  {i}. {model:20s} {score:.4f}{marker}")
        else:
            print("No plottable metrics found in results.")

except Exception as e:
    print(f"Model comparison chart skipped: {e}")
    import traceback
    traceback.print_exc()

## Feature Importance

Top features ranked by importance. Uses MDA (Mean Decrease in Accuracy) when available, falls back to model-reported importance.

In [ ]:
# =============================================================
# FEATURE IMPORTANCE: Top features by MDA or model importance
# =============================================================
try:
    import matplotlib.pyplot as plt
    import pandas as pd
    import numpy as np
    from pathlib import Path

    if "result" not in dir() or result is None or not result.success:
        print("No results available. Run Cell 5 first.")
    else:
        importance_found = False

        # Try to get feature importance from result object
        if hasattr(result, 'feature_importance') and result.feature_importance is not None:
            fi = result.feature_importance
            if isinstance(fi, dict):
                fi = pd.Series(fi)
            if isinstance(fi, pd.Series) and len(fi) > 0:
                fi = fi.sort_values(ascending=True).tail(20)
                fig, ax = plt.subplots(figsize=(10, max(6, len(fi) * 0.35)))
                fi.plot(kind='barh', ax=ax, color='#2196F3', edgecolor='black', linewidth=0.5)
                ax.set_title('Top 20 Features by Importance (MDA)', fontsize=13)
                ax.set_xlabel('Mean Decrease in Accuracy')
                ax.grid(True, axis='x', alpha=0.3)
                plt.tight_layout()
                plt.show()
                importance_found = True

        # Try to get per-model feature counts from bundles
        if hasattr(result, 'output_dir') and result.output_dir:
            bundle_dir = Path(result.output_dir) / "bundles"
            if bundle_dir.exists():
                import json as _json
                model_features = {}
                for bp in sorted(bundle_dir.glob("*/metadata.json")):
                    try:
                        meta = _json.loads(bp.read_text())
                        mname = meta.get("model_name", bp.parent.name)
                        n_feats = meta.get("n_features", meta.get("num_features", None))
                        if n_feats is not None:
                            model_features[mname] = int(n_feats)
                    except Exception:
                        pass

                if model_features:
                    mf_series = pd.Series(model_features).sort_values()
                    fig, ax = plt.subplots(figsize=(10, max(4, len(mf_series) * 0.4)))
                    mf_series.plot(kind='barh', ax=ax, color='#4CAF50', edgecolor='black', linewidth=0.5)
                    ax.set_title('Feature Count per Model (Per-Model Selection)', fontsize=13)
                    ax.set_xlabel('Number of Features')
                    ax.grid(True, axis='x', alpha=0.3)
                    plt.tight_layout()
                    plt.show()
                    importance_found = True

        if not importance_found:
            print("No feature importance data found in results or bundles.")
            print("Enable FEATURE_SELECTION_ENABLED=True in Cell 2 for MDA-based feature ranking.")

except Exception as e:
    print(f"Feature importance visualization skipped: {e}")
    import traceback
    traceback.print_exc()

## Backtest Results

Equity curve, drawdown chart, and key performance stats. Only displayed when `RUN_BACKTEST=True` in Cell 2.

In [ ]:
# =============================================================
# BACKTEST: Equity Curve + Drawdown + Stats
# =============================================================
try:
    import matplotlib.pyplot as plt
    import pandas as pd
    import numpy as np

    if "result" not in dir() or result is None or not result.success:
        print("No results available. Run the pipeline cell first.")
    elif not RUN_BACKTEST:
        print("Backtest not enabled. Set RUN_BACKTEST=True in Cell 2 to see equity curve.")
    elif not result.backtest_metrics:
        print("No backtest metrics found in results. The backtest may have failed.")
    else:
        # Try to get equity curve from result
        equity_curve = None
        if hasattr(result, 'equity_curve') and result.equity_curve is not None:
            equity_curve = result.equity_curve
        elif hasattr(result, 'backtest_equity') and result.backtest_equity is not None:
            equity_curve = result.backtest_equity

        if equity_curve is not None and len(equity_curve) > 1:
            if isinstance(equity_curve, pd.DataFrame):
                equity_curve = equity_curve.iloc[:, 0]

            fig, axes = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={'height_ratios': [3, 1]})

            # Equity curve
            axes[0].plot(equity_curve.index, equity_curve.values, linewidth=1.2, color='#1f77b4')
            axes[0].set_title('Backtest Equity Curve', fontsize=14)
            axes[0].set_ylabel('Portfolio Value')
            axes[0].grid(True, alpha=0.3)
            axes[0].axhline(equity_curve.iloc[0], color='gray', linestyle='--', alpha=0.5, label='Starting Value')
            axes[0].legend()

            # Drawdown
            running_max = equity_curve.cummax()
            drawdown = (equity_curve / running_max - 1) * 100
            axes[1].fill_between(drawdown.index, drawdown.values, alpha=0.4, color='red')
            axes[1].set_title('Drawdown (%)', fontsize=11)
            axes[1].set_ylabel('Drawdown %')
            axes[1].grid(True, alpha=0.3)

            plt.tight_layout()
            plt.show()
        else:
            print("Equity curve data not available for plotting.")

        # Print key stats including financial metrics
        bm = result.backtest_metrics
        print("\n" + "=" * 50)
        print("BACKTEST PERFORMANCE SUMMARY")
        print("=" * 50)
        stat_keys = [
            ("total_return_pct", "Total Return (%)"),
            ("sharpe_ratio", "Sharpe Ratio"),
            ("sortino_ratio", "Sortino Ratio"),
            ("calmar_ratio", "Calmar Ratio"),
            ("max_drawdown_pct", "Max Drawdown (%)"),
            ("profit_factor", "Profit Factor"),
            ("win_rate_pct", "Win Rate (%)"),
            ("expectancy", "Expectancy ($)"),
            ("total_trades", "Total Trades"),
            ("avg_trade_pct", "Avg Trade (%)"),
            ("avg_win_pct", "Avg Win (%)"),
            ("avg_loss_pct", "Avg Loss (%)"),
        ]
        for key, label in stat_keys:
            if key in bm:
                val = bm[key]
                if isinstance(val, float):
                    print(f"  {label:25s} {val:>10.4f}")
                else:
                    print(f"  {label:25s} {val:>10}")

        # Transaction cost summary
        cost_keys = [
            ("total_commission", "Total Commission ($)"),
            ("total_slippage", "Total Slippage ($)"),
        ]
        cost_found = [k for k, _ in cost_keys if k in bm]
        if cost_found:
            print(f"\n  {'--- Transaction Costs ---':^35}")
            for key, label in cost_keys:
                if key in bm:
                    print(f"  {label:25s} {bm[key]:>10.2f}")

except Exception as e:
    print(f"Backtest visualization skipped: {e}")
    import traceback
    traceback.print_exc()

## Interpreting Your Results

### What's a Good Score?
- **F1 > 0.35** is decent for 3-class financial prediction (buy/hold/sell)
- **F1 > 0.40** is good — your features have real predictive power
- **F1 > 0.50** is excellent — verify you're not overfitting or leaking data
- **F1 ≈ 0.33** means the model is ~random (1/3 chance for each class)

### What to Look For
- **Consistency across folds** — a model with 0.38 F1 on every fold is better than one with 0.50 on fold 1 and 0.20 on fold 2
- **Ensemble > best single model** — if the ensemble underperforms the best model, features may not be diverse enough
- **Walk-forward stability** — scores should be relatively stable across time windows (if using walk-forward mode)

### Red Flags
- **One model dramatically better than all others** — likely overfitting; check if it generalizes in walk-forward
- **All models score nearly the same** — features may not be predictive; try different feature sets or horizons
- **Test score >> train score** — data leakage; check purge/embargo settings
- **Neural models much worse than boosting** — may need more data or epochs (set MAX_EPOCHS ≥ 50)

### Next Steps
1. **Try different horizons** — HORIZONS = [5, 10, 20] to see which prediction window works best
2. **Enable feature selection** — set FEATURE_SELECTION_ENABLED=True to prune noise features
3. **Walk-forward validation** — set TRAINING_MODE="walk_forward" for more realistic evaluation
4. **Tune hyperparameters** — set OPTUNA_ENABLED=True, OPTUNA_TRIALS=50 for Bayesian optimization
5. **More data** — 1 week is minimal; 3-6 months gives much more reliable results

## Save & Export

Save results to Google Drive (Colab), export inference-only bundles, or download full results.

In [ ]:
# =============================================================
# CELL 8: GOOGLE DRIVE - Save results to Drive (Colab only)
# =============================================================
from pathlib import Path
import shutil

DRIVE_DEST = "ml_factory_results"

if IN_COLAB:
    try:
        from google.colab import drive

        drive.mount("/content/drive", force_remount=False)
        drive_root = Path("/content/drive/MyDrive") / DRIVE_DEST
        drive_root.mkdir(parents=True, exist_ok=True)

        print(f"Google Drive mounted. Save path: {drive_root}")

        # Copy results to Drive if they exist
        if "result" in dir() and result is not None and result.success:
            # Copy deploy artifact
            if result.deploy_path and Path(result.deploy_path).exists():
                dest = drive_root / EXPERIMENT_NAME / "deploy"
                if dest.exists():
                    shutil.rmtree(dest)
                shutil.copytree(result.deploy_path, dest)
                print(f"  Deploy artifact saved to Drive: {dest}")

            # Copy output dir
            if result.output_dir and Path(result.output_dir).exists():
                dest = drive_root / EXPERIMENT_NAME / "output"
                if dest.exists():
                    shutil.rmtree(dest)
                shutil.copytree(result.output_dir, dest)
                print(f"  Output dir saved to Drive: {dest}")

            print(f"\nResults persisted to Google Drive: {drive_root / EXPERIMENT_NAME}")
        else:
            print("No results to save yet. Run Cell 5 first, then re-run this cell.")

    except ImportError:
        print("google.colab not available. Running locally?")
    except Exception as e:
        print(f"Drive mount failed: {e}")
else:
    print("Not in Colab - skipping Drive mount. Results saved locally.")
    if "result" in dir() and result is not None and result.success:
        print(f"  Output: {result.output_dir}")
        if result.deploy_path:
            print(f"  Deploy: {result.deploy_path}")

In [ ]:
# =============================================================
# CELL 9: INFERENCE-ONLY EXPORT
# =============================================================
# Zip only the deploy/ or bundles/ directory for lightweight
# deployment. Excludes training logs, plots, and checkpoints.

from pathlib import Path
import shutil

if "result" not in dir() or result is None or not result.success:
    print("No successful result. Run Cell 5 first.")
else:
    output_dir = Path(result.output_dir) if result.output_dir else None
    deploy_dir = Path(result.deploy_path) if result.deploy_path else None

    # Prefer deploy/ artifact (self-contained), fall back to bundles/
    if deploy_dir and deploy_dir.exists():
        export_src = deploy_dir
        export_label = "deploy"
    elif output_dir:
        bundles_dir = output_dir / "bundles"
        if bundles_dir.exists():
            export_src = bundles_dir
            export_label = "bundles"
        else:
            export_src = None
            export_label = None
    else:
        export_src = None
        export_label = None

    if export_src is None:
        print("No deploy/ or bundles/ directory found to export.")
    else:
        zip_name = f"{EXPERIMENT_NAME}_inference_only"
        if IN_COLAB:
            zip_path = f"/content/{zip_name}"
        else:
            zip_path = str(output_dir.parent / zip_name) if output_dir else zip_name

        shutil.make_archive(zip_path, "zip", export_src)
        zip_file = Path(f"{zip_path}.zip")

        print("=" * 50)
        print("INFERENCE-ONLY EXPORT")
        print("=" * 50)
        print(f"  Source:   {export_src} ({export_label})")
        print(f"  Archive:  {zip_file}")
        print(f"  Size:     {zip_file.stat().st_size / 1e6:.1f} MB")

        # Compare with full output size
        if output_dir and output_dir.exists():
            full_size = sum(f.stat().st_size for f in output_dir.rglob("*") if f.is_file())
            infer_size = zip_file.stat().st_size
            print(f"\n  Full output:     {full_size / 1e6:.1f} MB")
            print(f"  Inference-only:  {infer_size / 1e6:.1f} MB")
            if full_size > 0:
                print(f"  Reduction:       {(1 - infer_size / full_size) * 100:.0f}%")

        # Auto-download in Colab
        if IN_COLAB:
            try:
                from google.colab import files
                files.download(str(zip_file))
            except Exception:
                print(f"\nManual download: files.download('{zip_file}')")

        print(f"\nTo use: unzip {zip_file.name} and load with load_deploy_artifact()")

In [ ]:
# =============================================================
# CELL 10: SAVE & DOWNLOAD RESULTS
# =============================================================
from pathlib import Path
import shutil

if "result" not in dir() or result is None or not result.success:
    print("No successful result to save.")
elif result.output_dir and Path(result.output_dir).exists():
    src_dir = Path(result.output_dir)

    if IN_COLAB:
        # Zip results for download
        zip_path = f"/content/{EXPERIMENT_NAME}_results"
        shutil.make_archive(zip_path, "zip", src_dir)
        print(f"Results zipped: {zip_path}.zip")
        print(f"Size: {Path(zip_path + '.zip').stat().st_size / 1e6:.1f} MB")

        # Auto-download in Colab
        try:
            from google.colab import files
            files.download(f"{zip_path}.zip")
        except Exception:
            print(f"\nManual download: files.download('{zip_path}.zip')")

        # Also try saving to Drive if mounted
        drive_path = Path("/content/drive/MyDrive/ml_factory_results")
        if drive_path.parent.exists():
            save_dest = drive_path / EXPERIMENT_NAME
            if save_dest.exists():
                shutil.rmtree(save_dest)
            shutil.copytree(src_dir, save_dest)
            print(f"\nAlso saved to Drive: {save_dest}")
    else:
        print(f"Results at: {src_dir}")
        n_files = sum(1 for f in src_dir.rglob("*") if f.is_file())
        total_mb = sum(f.stat().st_size for f in src_dir.rglob("*") if f.is_file()) / 1e6
        print(f"Files: {n_files}, Size: {total_mb:.1f} MB")
else:
    print("No output directory found.")